### Extended Data Figure 5 — Input and output directories


In [ ]:
NOTEBOOK_NAME = 'AAA_final_figs5'
from pathlib import Path
import os
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'pipelines/run.py').is_file()))
INPUT_ROOT = Path(os.environ.get('CIPHER_ANALYSIS_ROOT', str(ROOT / 'figure_data'))).resolve()
OUTPUT_ROOT = ROOT / 'figure_outputs' / NOTEBOOK_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_ROOT)


### Extended Data Figure 5c — Representative ECG risk lookup


In [ ]:
import os
import pandas as pd
import numpy as np
PID = int(os.environ["CIPHER_EXAMPLE_PID"])
CSV_PATH = 'code_02_02_save/ecg_20260203_1003/preds/CHD_ecg_transformer_h256_l3_test_best_pred.csv'
df = pd.read_csv(str(OUTPUT_ROOT / CSV_PATH if (OUTPUT_ROOT / CSV_PATH).exists() else INPUT_ROOT / CSV_PATH))
risk_min = df['risk'].min()
risk_max = df['risk'].max()
df['risk_norm'] = (df['risk'] - risk_min) / (risk_max - risk_min)
row = df[df['ID'] == PID]
if len(row) == 0:
    print(f'PID {PID} not found')
else:
    risk_raw = row['risk'].values[0]
    risk_norm = row['risk_norm'].values[0]
    time_val = row['time'].values[0]
    event_val = row['event'].values[0]
    print('\n==============================')
    print(f'PID            : {PID}')
    print(f'Raw risk       : {risk_raw:.6f}')
    print(f'Normalized risk: {risk_norm:.6f}')
    print(f'Time           : {time_val:.6f}')
    print(f'Event          : {event_val}')
    print('==============================\n')


### Supporting analysis — Fundus/proteomic protein heatmaps


In [ ]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
try:
    import gseapy as gp
    HAS_GSEAPY = True
except Exception:
    HAS_GSEAPY = False
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.compression'] = 0
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['text.usetex'] = False
BASE = '.'
CORR_DIR = os.path.join(BASE, 'code_3_3_save/risk_feature_correlation')
OUT_DIR = os.path.join('fig3_final', 'protein_heatmap_Fundus_vs_Proteomics_same_direction_top_bottom')
os.makedirs(OUT_DIR, exist_ok=True)
COLORBAR_DIR = os.path.join(OUT_DIR, 'colorbar')
os.makedirs(COLORBAR_DIR, exist_ok=True)
GO_DIR = os.path.join(OUT_DIR, 'GO_enrichment')
os.makedirs(GO_DIR, exist_ok=True)
CORR_OUT_DIR = os.path.join(OUT_DIR, 'computed_proteomics_corr')
os.makedirs(CORR_OUT_DIR, exist_ok=True)
ENDPOINTS = ['CHD', 'istroke', 'ASCVD']
ENDPOINT_ALIASES = {'CHD': ['CHD'], 'istroke': ['istroke', 'iStroke', 'ISTROKE'], 'ASCVD': ['ASCVD', 'ICVD']}
P_CUTOFF = 1
TOP_N = 100
BOTTOM_N = 100
SMOOTH_SIGMA_X = 5.0
Z_CLIP = 2.0
FIG_DPI = 300
GO_LIBRARIES = ['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'GO_Cellular_Component_2023']
RISK_PRED = {'Fundus': {'pred_dir': os.path.join(BASE, 'code_02_02_save/fundus_20251229_1519/preds'), 'patterns': ['{endpoint}_fundus_vitsmallFT_test_best_pred.csv', '{endpoint}_fundus_vitsmallFT_test_pred.csv']}, 'Proteomics': {'pred_dir': os.path.join(BASE, 'code_02_02_save/soma_20260202_2006/preds'), 'patterns': ['{endpoint}_soma_TR_test_best_pred.csv']}}
FEATURE_INPUT = {'input_dir': os.path.join(BASE, 'code_3_3_save/soma_single'), 'pattern': 'CHD_soma_TR_best_IG_input.csv', 'value_col': 'ig_abs'}
HIGHLIGHT_GENES = set(['ANGPTL3', 'ADIPOQ', 'VEGFC', 'PLAT', 'FAS', 'IL1RAP', 'FGR', 'CLEC3B', 'LDLR', 'SORT1', 'MMP9', 'PLA2G2A', 'AXL', 'SLAMF8', 'TNFRSF8', 'JAM3', 'ICAM4', 'CD59', 'TIMP3', 'TNC', 'CCL3', 'CCL18', 'CCL11', 'MMP3', 'GDF15', 'CST3', 'BCL2L11', 'ACVRL1', 'CLEC14A', 'CD93', 'DDR1', 'MMP12', 'EPO', 'HSPB1', 'WFDC2'])

def get_endpoint_aliases(endpoint):
    return ENDPOINT_ALIASES.get(endpoint, [endpoint])

def find_existing_file(candidates, description='file'):
    for path in candidates:
        if os.path.exists(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path)):
            return path
    msg = f'\n[ERROR] Cannot find {description}. Tried:\n'
    msg += '\n'.join([f'  - {p}' for p in candidates])
    raise FileNotFoundError(msg)

def rank_norm(x):
    r = pd.Series(x).rank(method='average').values
    return (r - 1) / (len(r) - 1 + 1e-08)

def clean_protein_name(x):
    x = str(x)
    if x.startswith('seq.'):
        x = x.replace('seq.', '', 1)
    return x

def load_risk(endpoint, risk_model):
    cfg = RISK_PRED[risk_model]
    candidates = []
    for ep in get_endpoint_aliases(endpoint):
        for pat in cfg['patterns']:
            candidates.append(os.path.join(cfg['pred_dir'], pat.format(endpoint=ep)))
    path = find_existing_file(candidates, description=f'{endpoint} {risk_model} prediction file')
    print(f'[Risk] {endpoint} {risk_model}: {path}')
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'pid'})
    if 'pid' not in df.columns:
        raise ValueError(f'{path} must contain pid or ID column.')
    if 'risk' not in df.columns:
        raise ValueError(f'{path} must contain risk column.')
    keep_cols = ['pid', 'risk']
    for c in ['time', 'event']:
        if c in df.columns:
            keep_cols.append(c)
    df = df[keep_cols].copy()
    df['pid'] = df['pid'].astype(int)
    df['risk'] = pd.to_numeric(df['risk'], errors='coerce')
    df = df.dropna(subset=['risk'])
    df['risk_norm'] = rank_norm(df['risk'].values)
    return df

def load_fundus_corr(endpoint):
    candidates = []
    for ep in get_endpoint_aliases(endpoint):
        candidates.extend([os.path.join(CORR_DIR, f'{ep}_Fundus_vs_SomaProtein_pearson.csv'), os.path.join(CORR_DIR, f'{ep}_fundus_vs_SomaProtein_pearson.csv')])
    path = find_existing_file(candidates, description=f'{endpoint} Fundus vs SomaProtein pearson correlation file')
    print(f'[Corr] {endpoint} Fundus: {path}')
    corr = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    required = ['feature', 'pearson_r', 'pval']
    for c in required:
        if c not in corr.columns:
            raise ValueError(f'{path} must contain column: {c}')
    corr = corr[corr['pval'] < P_CUTOFF].copy()
    corr = corr.dropna(subset=['feature', 'pearson_r', 'pval'])
    corr['feature'] = corr['feature'].astype(str)
    corr['pearson_r'] = pd.to_numeric(corr['pearson_r'], errors='coerce')
    corr['pval'] = pd.to_numeric(corr['pval'], errors='coerce')
    corr = corr.dropna(subset=['pearson_r', 'pval'])
    if 'abs_pearson_r' not in corr.columns:
        corr['abs_pearson_r'] = corr['pearson_r'].abs()
    if 'direction' not in corr.columns:
        corr['direction'] = np.where(corr['pearson_r'] >= 0, 'positive', 'negative')
    return corr

def load_soma_long(endpoint):
    candidates = []
    for ep in get_endpoint_aliases(endpoint):
        candidates.append(os.path.join(FEATURE_INPUT['input_dir'], FEATURE_INPUT['pattern'].format(endpoint=ep)))
    path = find_existing_file(candidates, description=f'{endpoint} soma IG input file')
    print(f'[Protein matrix] {endpoint}: {path}')
    df = pd.read_csv(str(OUTPUT_ROOT / path if (OUTPUT_ROOT / path).exists() else INPUT_ROOT / path))
    for c in ['pid', 'feature', FEATURE_INPUT['value_col']]:
        if c not in df.columns:
            raise ValueError(f'{path} must contain column: {c}')
    df['pid'] = df['pid'].astype(int)
    df['feature'] = df['feature'].astype(str)
    df[FEATURE_INPUT['value_col']] = pd.to_numeric(df[FEATURE_INPUT['value_col']], errors='coerce')
    df = df.dropna(subset=[FEATURE_INPUT['value_col']])
    return df

def compute_proteomics_corr_from_risk(endpoint, soma_long, risk_proteomics):
    risk_df = risk_proteomics[['pid', 'risk']].copy()
    risk_df['pid'] = risk_df['pid'].astype(int)
    risk_df['risk'] = pd.to_numeric(risk_df['risk'], errors='coerce')
    risk_df = risk_df.dropna(subset=['risk'])
    df = soma_long.merge(risk_df, on='pid', how='inner')
    value_col = FEATURE_INPUT['value_col']
    results = []
    for (feature, g) in df.groupby('feature', sort=False):
        tmp = g[['risk', value_col]].dropna()
        n = len(tmp)
        if n < 5:
            r = np.nan
            p = np.nan
        else:
            x = tmp['risk'].astype(float).values
            y = tmp[value_col].astype(float).values
            if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
                r = np.nan
                p = np.nan
            else:
                (r, p) = pearsonr(x, y)
        results.append({'endpoint': endpoint, 'risk_model': 'Proteomics', 'feature_set': 'SomaProtein', 'feature': feature, 'n': n, 'pearson_r': r, 'pval': p})
    corr = pd.DataFrame(results)
    corr = corr.dropna(subset=['pearson_r', 'pval']).copy()
    if len(corr) > 0:
        corr['fdr'] = multipletests(corr['pval'].values, method='fdr_bh')[1]
    else:
        corr['fdr'] = []
    corr['abs_pearson_r'] = corr['pearson_r'].abs()
    corr['direction'] = np.where(corr['pearson_r'] >= 0, 'positive', 'negative')
    corr = corr.sort_values('abs_pearson_r', ascending=False).reset_index(drop=True)
    out_csv = os.path.join(CORR_OUT_DIR, f'{endpoint}_Proteomics_vs_SomaProtein_pearson_computed.csv')
    corr.to_csv(out_csv, index=False)
    print(f'[Saved computed corr] {out_csv}')
    corr = corr[corr['pval'] < P_CUTOFF].copy()
    return corr

def get_top_and_bottom(corr):
    top_pos = corr[corr['pearson_r'] > 0].sort_values('pearson_r', ascending=False).head(TOP_N).copy()
    top_neg = corr[corr['pearson_r'] < 0].sort_values('pearson_r', ascending=True).head(BOTTOM_N).copy()
    return (top_pos, top_neg)

def load_common_same_direction_features(endpoint, soma_long, risk_proteomics):
    corr_fundus = load_fundus_corr(endpoint)
    corr_prot = compute_proteomics_corr_from_risk(endpoint=endpoint, soma_long=soma_long, risk_proteomics=risk_proteomics)
    (fundus_top, fundus_bottom) = get_top_and_bottom(corr_fundus)
    (prot_top, prot_bottom) = get_top_and_bottom(corr_prot)
    both_top_features = set(fundus_top['feature'].astype(str)) & set(prot_top['feature'].astype(str))
    both_bottom_features = set(fundus_bottom['feature'].astype(str)) & set(prot_bottom['feature'].astype(str))
    selected_features = sorted(both_top_features | both_bottom_features)
    print(f'\n=== {endpoint} same-direction selection ===')
    print(f'Fundus p<{P_CUTOFF}:', len(corr_fundus))
    print(f'Proteomics p<{P_CUTOFF}:', len(corr_prot))
    print(f'Fundus top{TOP_N}:', len(fundus_top))
    print(f'Proteomics top{TOP_N}:', len(prot_top))
    print(f'Fundus bottom{BOTTOM_N}:', len(fundus_bottom))
    print(f'Proteomics bottom{BOTTOM_N}:', len(prot_bottom))
    print('Both top positive:', len(both_top_features))
    print('Both bottom negative:', len(both_bottom_features))
    print('Selected same-direction features:', len(selected_features))
    if len(selected_features) == 0:
        return ([], None)
    fundus_sel = pd.concat([fundus_top, fundus_bottom], axis=0)
    prot_sel = pd.concat([prot_top, prot_bottom], axis=0)
    score = fundus_sel[fundus_sel['feature'].isin(selected_features)][['feature', 'pearson_r', 'pval']].rename(columns={'pearson_r': 'fundus_r', 'pval': 'fundus_p'}).merge(prot_sel[prot_sel['feature'].isin(selected_features)][['feature', 'pearson_r', 'pval']].rename(columns={'pearson_r': 'proteomics_r', 'pval': 'proteomics_p'}), on='feature', how='inner')
    score['same_direction_group'] = np.where((score['fundus_r'] > 0) & (score['proteomics_r'] > 0), 'both_positive', 'both_negative')
    score['mean_abs_r'] = (score['fundus_r'].abs() + score['proteomics_r'].abs()) / 2
    score['mean_r'] = (score['fundus_r'] + score['proteomics_r']) / 2
    score['group_order'] = score['same_direction_group'].map({'both_positive': 0, 'both_negative': 1})
    score_pos = score[score['same_direction_group'] == 'both_positive'].sort_values('mean_abs_r', ascending=False).copy()
    score_neg = score[score['same_direction_group'] == 'both_negative'].sort_values('mean_abs_r', ascending=False).copy()
    score = pd.concat([score_pos, score_neg], axis=0).reset_index(drop=True)
    ordered_features = score['feature'].astype(str).tolist()
    return (ordered_features, score)

def make_expression_matrix(soma_long, features, pids_keep):
    df = soma_long[soma_long['feature'].isin(features) & soma_long['pid'].isin(set(pids_keep))].copy()
    mat = df.pivot_table(index='feature', columns='pid', values=FEATURE_INPUT['value_col'], aggfunc='mean')
    return mat

def zscore_rows(mat):
    arr = mat.values.astype(float)
    mean = np.nanmean(arr, axis=1, keepdims=True)
    std = np.nanstd(arr, axis=1, keepdims=True)
    z = (arr - mean) / (std + 1e-08)
    z = np.clip(z, -Z_CLIP, Z_CLIP)
    return z

def smooth_rows_only_x(z):
    return gaussian_filter1d(z, sigma=SMOOTH_SIGMA_X, axis=1, mode='nearest')

def prepare_matrix_for_model(mat, risk_df, order_direction):
    if order_direction == 'descending':
        risk_df = risk_df.sort_values('risk', ascending=False).copy()
    elif order_direction == 'ascending':
        risk_df = risk_df.sort_values('risk', ascending=True).copy()
    else:
        raise ValueError("order_direction must be 'ascending' or 'descending'")
    ordered_pids = risk_df['pid'].astype(int).tolist()
    keep_cols = [p for p in ordered_pids if p in mat.columns]
    mat_model = mat.loc[:, keep_cols]
    z = zscore_rows(mat_model)
    z = smooth_rows_only_x(z)
    risk_bar = risk_df.set_index('pid').loc[mat_model.columns]['risk_norm'].values.reshape(1, -1)
    return (z, risk_bar, mat_model.columns.tolist())

def save_colorbars():
    (fig, ax) = plt.subplots(figsize=(1.2, 4))
    norm = mpl.colors.Normalize(vmin=-Z_CLIP, vmax=Z_CLIP)
    sm = mpl.cm.ScalarMappable(cmap='RdBu_r', norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label('Protein z-score', fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    for ext in ['pdf', 'png', 'svg']:
        fig.savefig(os.path.join(COLORBAR_DIR, f'protein_zscore_colorbar.{ext}'), dpi=FIG_DPI, transparent=False, facecolor='white', bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    (fig, ax) = plt.subplots(figsize=(1.2, 4))
    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    sm = mpl.cm.ScalarMappable(cmap='gray_r', norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label('Normalized risk', fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    for ext in ['pdf', 'png', 'svg']:
        fig.savefig(os.path.join(COLORBAR_DIR, f'risk_colorbar.{ext}'), dpi=FIG_DPI, transparent=False, facecolor='white', bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    (fig, ax) = plt.subplots(figsize=(1.2, 4))
    norm = mpl.colors.Normalize(vmin=-0.3, vmax=0.3)
    sm = mpl.cm.ScalarMappable(cmap='PuOr_r', norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label('Pearson correlation', fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    for ext in ['pdf', 'png', 'svg']:
        fig.savefig(os.path.join(COLORBAR_DIR, f'correlation_colorbar.{ext}'), dpi=FIG_DPI, transparent=False, facecolor='white', bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print('Saved colorbars:', COLORBAR_DIR)

def run_go_enrichment(endpoint, score_df):
    if not HAS_GSEAPY:
        print('[GO skip] gseapy not installed.')
        return
    go_out = os.path.join(GO_DIR, endpoint)
    os.makedirs(go_out, exist_ok=True)
    groups = {'both_positive': score_df[score_df['same_direction_group'] == 'both_positive'], 'both_negative': score_df[score_df['same_direction_group'] == 'both_negative']}
    for (group_name, gdf) in groups.items():
        genes = sorted(set([clean_protein_name(x) for x in gdf['feature'].astype(str).tolist()]))
        genes = [g for g in genes if g and g.lower() != 'nan']
        gene_table = os.path.join(go_out, f'{group_name}_genes.csv')
        pd.DataFrame({'gene': genes}).to_csv(gene_table, index=False)
        print(f'[GO] {endpoint} {group_name}: n_genes={len(genes)}')
        if len(genes) < 5:
            print(f'[GO skip] {endpoint} {group_name}: too few genes')
            continue
        group_out = os.path.join(go_out, group_name)
        os.makedirs(group_out, exist_ok=True)
        for lib in GO_LIBRARIES:
            try:
                enr = gp.enrichr(gene_list=genes, gene_sets=lib, organism='Human', outdir=group_out, cutoff=1.0, no_plot=True)
                if enr.results is not None and len(enr.results) > 0:
                    res = enr.results.copy()
                    res.to_csv(os.path.join(group_out, f'{lib}_enrichr_results.csv'), index=False)
                    if 'Adjusted P-value' in res.columns:
                        res = res.sort_values('Adjusted P-value', ascending=True).head(15).copy()
                        res['minus_log10_fdr'] = -np.log10(res['Adjusted P-value'] + 1e-300)
                        (fig, ax) = plt.subplots(figsize=(6, 4))
                        ax.barh(np.arange(len(res)), res['minus_log10_fdr'].values)
                        ax.set_yticks(np.arange(len(res)))
                        ax.set_yticklabels(res['Term'].values, fontsize=7)
                        ax.invert_yaxis()
                        ax.set_xlabel('-log10(FDR)')
                        ax.set_title(f'{endpoint} {group_name}\n{lib}', fontsize=10)
                        for ext in ['pdf', 'png', 'svg']:
                            fig.savefig(os.path.join(group_out, f'{lib}_top15_barplot.{ext}'), dpi=FIG_DPI, transparent=False, facecolor='white', bbox_inches='tight', pad_inches=0.05)
                        plt.close(fig)
            except Exception as e:
                print(f'[GO failed] {endpoint} {group_name} {lib}: {e}')

def plot_combined(endpoint, features, score_df, risk_fundus, risk_proteomics, mat):
    mat = mat.loc[[f for f in features if f in mat.index]]
    if mat.empty:
        print(f'[Skip] {endpoint}: empty same-direction matrix')
        return
    score_df = score_df.set_index('feature').loc[mat.index].reset_index()
    (z_fundus, risk_bar_fundus, cols_fundus) = prepare_matrix_for_model(mat, risk_fundus, order_direction='descending')
    (z_prot, risk_bar_prot, cols_prot) = prepare_matrix_for_model(mat, risk_proteomics, order_direction='ascending')
    fundus_corr = score_df['fundus_r'].values.reshape(-1, 1)
    prot_corr = score_df['proteomics_r'].values.reshape(-1, 1)
    corr_lim = max(0.05, np.nanmax(np.abs(score_df[['fundus_r', 'proteomics_r']].values)))
    n_features = mat.shape[0]
    n_fundus = len(cols_fundus)
    n_prot = len(cols_prot)
    fig_h = max(2.8, n_features * 0.18)
    fig_w = 7.5
    fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=False)
    gs = fig.add_gridspec(2, 5, width_ratios=[0.28, 5.0, 1.7, 5.0, 0.28], height_ratios=[0.45, 5.0], wspace=0.04, hspace=0.08)
    ax_risk_fundus = fig.add_subplot(gs[0, 1])
    ax_risk_prot = fig.add_subplot(gs[0, 3])
    ax_corr_fundus = fig.add_subplot(gs[1, 0])
    ax_heat_fundus = fig.add_subplot(gs[1, 1])
    ax_names = fig.add_subplot(gs[1, 2])
    ax_heat_prot = fig.add_subplot(gs[1, 3])
    ax_corr_prot = fig.add_subplot(gs[1, 4])
    ax_risk_fundus.imshow(risk_bar_fundus, aspect='auto', cmap='gray_r', vmin=0, vmax=1, interpolation='nearest', rasterized=True)
    ax_risk_fundus.set_title('Fundus risk', fontsize=12, pad=8)
    ax_risk_fundus.set_xticks([])
    ax_risk_fundus.set_yticks([])
    ax_risk_prot.imshow(risk_bar_prot, aspect='auto', cmap='gray_r', vmin=0, vmax=1, interpolation='nearest', rasterized=True)
    ax_risk_prot.set_title('Proteomics risk', fontsize=12, pad=8)
    ax_risk_prot.set_xticks([])
    ax_risk_prot.set_yticks([])
    ax_corr_fundus.imshow(fundus_corr, aspect='auto', cmap='PuOr_r', vmin=-corr_lim, vmax=corr_lim, interpolation='nearest', rasterized=True)
    ax_corr_fundus.set_xticks([])
    ax_corr_fundus.set_yticks([])
    ax_corr_fundus.set_ylabel('Fundus\ncorr.', fontsize=8)
    ax_corr_prot.imshow(prot_corr, aspect='auto', cmap='PuOr_r', vmin=-corr_lim, vmax=corr_lim, interpolation='nearest', rasterized=True)
    ax_corr_prot.set_xticks([])
    ax_corr_prot.set_yticks([])
    ax_corr_prot.set_ylabel('Proteomics\ncorr.', fontsize=8)
    ax_corr_prot.yaxis.set_label_position('right')
    ax_heat_fundus.imshow(z_fundus, aspect='auto', cmap='RdBu_r', vmin=-Z_CLIP, vmax=Z_CLIP, interpolation='nearest', rasterized=True)
    ax_heat_fundus.set_xticks([])
    ax_heat_fundus.set_yticks([])
    ax_heat_prot.imshow(z_prot, aspect='auto', cmap='RdBu_r', vmin=-Z_CLIP, vmax=Z_CLIP, interpolation='nearest', rasterized=True)
    ax_heat_prot.set_xticks([])
    ax_heat_prot.set_yticks([])
    ax_names.set_xlim(0, 1)
    ax_names.set_ylim(n_features, 0)
    ax_names.axis('off')
    for (i, feat) in enumerate(mat.index.tolist()):
        name = clean_protein_name(feat)
        color = 'red' if name in HIGHLIGHT_GENES else 'black'
        ax_names.text(0.5, i + 0.5, name, fontsize=8, color=color, ha='center', va='center', fontstyle='italic')
    for ax in [ax_risk_fundus, ax_risk_prot, ax_corr_fundus, ax_heat_fundus, ax_names, ax_heat_prot, ax_corr_prot]:
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)
    out_prefix = os.path.join(OUT_DIR, f'{endpoint}_same_direction_Fundus_left_Proteomics_right_top{TOP_N}_bottom{BOTTOM_N}_corrbar')
    fig.subplots_adjust(left=0.04, right=0.98, top=0.86, bottom=0.08)
    for ext in ['pdf', 'png', 'svg']:
        fig.savefig(out_prefix + f'.{ext}', dpi=FIG_DPI, transparent=False, facecolor='white', edgecolor='white', bbox_inches='tight', pad_inches=0.08)
    plt.close(fig)
    print('Saved:', out_prefix + '.pdf')
    print('Saved:', out_prefix + '.png')
    print('Saved:', out_prefix + '.svg')
    print(f'n_features={n_features}, n_fundus_samples={n_fundus}, n_proteomics_samples={n_prot}')

def main():
    save_colorbars()
    for endpoint in ENDPOINTS:
        print('\n====================================================')
        print(f'Processing endpoint: {endpoint}')
        print('====================================================')
        risk_fundus = load_risk(endpoint, 'Fundus')
        risk_proteomics = load_risk(endpoint, 'Proteomics')
        soma_long = load_soma_long(endpoint)
        (features, score_df) = load_common_same_direction_features(endpoint=endpoint, soma_long=soma_long, risk_proteomics=risk_proteomics)
        if len(features) == 0:
            print(f'[Skip] No same-direction features for {endpoint}')
            continue
        run_go_enrichment(endpoint, score_df)
        all_pids = sorted(set(risk_fundus['pid'].astype(int).tolist()) | set(risk_proteomics['pid'].astype(int).tolist()))
        mat = make_expression_matrix(soma_long=soma_long, features=features, pids_keep=all_pids)
        if mat.empty:
            print(f'[Skip] Empty expression matrix for {endpoint}')
            continue
        keep_features = [f for f in features if f in mat.index]
        mat = mat.loc[keep_features]
        score_out = os.path.join(OUT_DIR, f'{endpoint}_same_direction_Fundus_Proteomics_top{TOP_N}_bottom{BOTTOM_N}_protein_table.csv')
        score_df.to_csv(score_out, index=False)
        print('Saved selected protein table:', score_out)
        plot_combined(endpoint=endpoint, features=keep_features, score_df=score_df, risk_fundus=risk_fundus, risk_proteomics=risk_proteomics, mat=mat)
    print('\nAll done.')
    print('Output folder:', OUT_DIR)
    print('Colorbar folder:', COLORBAR_DIR)
    print('Computed proteomics correlation folder:', CORR_OUT_DIR)
if __name__ == '__main__':
    main()


### the protein heatmap analysis — Inspect correlation files


In [ ]:
print([p.name for p in (INPUT_ROOT / 'data/private/risk_feature_correlation').iterdir()])


### the protein heatmap analysis — Preview a correlation table


In [ ]:
print('\n'.join((INPUT_ROOT / 'data/private/CHD_Fundus_vs_SomaProtein_pearson.csv').read_text().splitlines()[:10]))
